In [ ]:
"""
Leave-One-Site-Out (LOSO) ML Pipeline  — v4
============================================
5 models × 14 sites × 5-fold/10-repeat CV

Changes from v3
---------------
1. Training bar height : mean across ALL 50 CV fold scores (was best-repeat mean).
2. Training error bars : asymmetric min–max range [mean-min, max-mean] across all
                         50 CV fold scores (was ± 1 SD). Naturally bounded in [0, 1].
3. Test error bars     : asymmetric min–max range [mean-min, max-mean] across 1000
                         bootstrap resamples (was ± 1 SD). Naturally bounded in [0, 1].
4. Legend and annotations updated to reflect new method.

Produces 5 figures (one per model), each with 4 subplots (Accuracy, Precision,
Recall, F1).  Each subplot has 14 grouped bars (Training CV | Test), both with
asymmetric min–max error bars.
"""

# ============================================================================
# IMPORTS
# ============================================================================
import os, gc, random, warnings, copy
os.environ["TF_DETERMINISTIC_OPS"] = "1"
# Run from the notebooks/ directory OR set kernel working directory to HAB_Phase1/
import sys, pathlib

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
plt.ioff()

from scipy.stats import sem as _sem_unused  # kept for import compatibility; not used

from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from xgboost                 import XGBClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.metrics         import (
    accuracy_score, precision_score, recall_score, f1_score, make_scorer
)
from sklearn.preprocessing   import StandardScaler

import tensorflow as tf
import tensorflow.keras as keras

warnings.filterwarnings("ignore")

# ============================================================================
# CONFIGURATION  — edit these paths / columns as needed
# ============================================================================
PROJECT_ROOT = pathlib.Path().resolve()
while PROJECT_ROOT.name != "HAB_Phase1":
    PROJECT_ROOT = PROJECT_ROOT.parent
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Could not find HAB_Phase1 project root")
    
sys.path.insert(0, str(PROJECT_ROOT))

from config import DATA_PATH, OUTPUT_DIRS

# DATA_PATH    = "Data/UCDavis_Data_for_Analysis2_220days_ToxinProdMicrocystis.csv"
# OUTPUT_DIR   = "C:/Temp/LOSO_Results3"
OUTPUT_DIR = str(OUTPUT_DIRS["06"])  # change for each notebook
DATA_PATH  = str(DATA_PATH)

TARGET_COL   = "ToxinProdc_Microcystis_CC"
SITE_COL     = "Site_Name"
DATE_COL     = "Date"
THRESHOLD    = 4000          # binarisation threshold for the target column
RANDOM_STATE = 42

FEATURES = ["Q", "Q_1m", "SpecCond.", "OrgC", "OrgN", "PO4",
            "WaterTemp", "DO", "pH"]

# ---- CV settings ------------------------------------------------------------
CV_N_SPLITS  = 5
CV_N_REPEATS = 10   # 50 total folds per LOSO iteration

# ---- Bootstrap settings -----------------------------------------------------
BOOTSTRAP_N  = 1000   # resamples for test SD

# ---- ANN hyperparameters ----------------------------------------------------
ANN_HIDDEN_LAYERS = 3
ANN_ACTIVATION    = "elu"
ANN_NEURONS       = 128
ANN_LEARNING_RATE = 0.0081
ANN_DROPOUT_RATE  = 0.25
ANN_EPOCHS        = 1000
ANN_BATCH_SIZE    = 32
ANN_VAL_SPLIT     = 0.2
ANN_PATIENCE      = 30

ALL_MODELS = ["Logistic Regression", "Decision Tree", "XGBoost",
              "Random Forest", "ANN"]

METRICS = ["accuracy", "precision", "recall", "f1"]

# ---- Colours ----------------------------------------------------------------
MODEL_COLOR = {
    "Logistic Regression": "#9467BD",
    "Decision Tree":       "#8C564B",
    "Random Forest":       "#1C7293",
    "XGBoost":             "#B85042",
    "ANN":                 "#E8A838",
}
TRAIN_ALPHA = 0.85
TEST_ALPHA  = 0.55


# ============================================================================
# HELPERS
# ============================================================================
def set_global_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def savefig(fig, subfolder: str, filename: str):
    """Save figure as both PNG and EPS under OUTPUT_DIR/figures/<subfolder>/."""
    folder = os.path.abspath(os.path.join(OUTPUT_DIR, "figures", subfolder))
    os.makedirs(folder, exist_ok=True)
    for ext in ("png", "eps"):
        path = os.path.join(folder, f"{filename}.{ext}")
        try:
            fig.savefig(path, dpi=150, bbox_inches="tight",
                        format=ext if ext == "eps" else None)
            print(f"  Saved → {path}")
        except Exception as e:
            print(f"  [WARN] Could not save {path}: {e}")


def setup_dirs():
    for sub in ("figures/loso_performance", "tables", "logs"):
        os.makedirs(os.path.join(OUTPUT_DIR, sub), exist_ok=True)
    print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")


def safe_metrics(y_true, y_pred) -> dict:
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
    }


# ============================================================================
# CI HELPERS  — v4: mean + asymmetric min–max range, naturally bounded in [0,1]
# ============================================================================
def mean_minmax_from_scores(scores) -> tuple[float, float, float]:
    """
    Returns (mean, lower_err, upper_err) where:
      lower_err = mean - min   (how far the bar extends downward)
      upper_err = max - mean   (how far the bar extends upward)
    Both are >= 0 and the whiskers stay within [0, 1] by construction.
    """
    arr       = np.asarray(scores, dtype=float)
    mean      = float(arr.mean())
    lower_err = mean - float(arr.min())
    upper_err = float(arr.max()) - mean
    return mean, lower_err, upper_err


def bootstrap_minmax(y_true, y_pred,
                     n_bootstrap: int = BOOTSTRAP_N,
                     seed: int = RANDOM_STATE) -> dict:
    """
    Bootstrap mean + asymmetric min–max range on Accuracy, Precision, Recall, F1
    for a single left-out test set.

    Returns dict[metric] = {"mean": float, "lower_err": float, "upper_err": float}
    where lower_err = mean - min, upper_err = max - mean (both >= 0, bounded in [0,1]).
    """
    rng    = np.random.default_rng(seed)
    n      = len(y_true)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    boot_scores = {m: [] for m in METRICS}
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        yt, yp = y_true[idx], y_pred[idx]
        if len(np.unique(yt)) < 2:
            continue
        boot_scores["accuracy"].append(accuracy_score(yt, yp))
        boot_scores["precision"].append(precision_score(yt, yp, zero_division=0))
        boot_scores["recall"].append(recall_score(yt, yp, zero_division=0))
        boot_scores["f1"].append(f1_score(yt, yp, zero_division=0))

    result = {}
    for m, vals in boot_scores.items():
        arr = np.asarray(vals, dtype=float)
        if len(arr) < 2:
            result[m] = {"mean": float(np.nan), "lower_err": 0.0, "upper_err": 0.0}
        else:
            mean = float(arr.mean())
            result[m] = {
                "mean":      mean,
                "lower_err": mean - float(arr.min()),
                "upper_err": float(arr.max()) - mean,
            }
    return result


# ============================================================================
# DATA LOADER
# ============================================================================
def load_data():
    df = pd.read_csv(DATA_PATH)
    df["label"] = (df[TARGET_COL] >= THRESHOLD).astype(int)
    df = df.dropna(subset=FEATURES + ["label", SITE_COL])
    X     = df[FEATURES].copy()
    y     = df["label"].values
    sites = df[SITE_COL].values
    return X, y, sites, df


# ============================================================================
# MODEL BUILDERS
# ============================================================================
def build_lr():
    return LogisticRegression(
        warm_start=False, tol=0.0001, solver="saga", penalty="l1",
        max_iter=1000, fit_intercept=True, class_weight="balanced",
        C=1, random_state=RANDOM_STATE,
    )

def build_dt():
    return DecisionTreeClassifier(
        splitter="random", min_samples_split=5, min_samples_leaf=2,
        min_impurity_decrease=0.0, max_leaf_nodes=None, max_features=0.5,
        max_depth=5, criterion="entropy", class_weight="balanced",
        ccp_alpha=0.01,
    )

def build_xgb():
    return XGBClassifier(
        subsample=0.8, scale_pos_weight=1.0, n_estimators=200,
        min_child_weight=3, max_depth=2, learning_rate=0.05,
        reg_lambda=1.0, gamma=0.3, colsample_bytree=0.5,
        alpha=0.1, colsample_bylevel=0.8278, colsample_bynode=0.7501,
        max_delta_step=0,
        random_state=RANDOM_STATE, verbosity=0, n_jobs=1,
        objective="binary:logistic",
    )

def build_rf():
    return RandomForestClassifier(
        n_estimators=300, max_depth=5, min_samples_split=5,
        min_samples_leaf=8, max_features=0.3, bootstrap=True,
        class_weight="balanced", criterion="gini", ccp_alpha=0.0001,
        min_impurity_decrease=0.01627, min_weight_fraction_leaf=0.01946,
        max_leaf_nodes=57, random_state=RANDOM_STATE,
    )

SKLEARN_BUILDERS = {
    "Logistic Regression": build_lr,
    "Decision Tree":       build_dt,
    "XGBoost":             build_xgb,
    "Random Forest":       build_rf,
}


# ============================================================================
# ANN WRAPPER
# ============================================================================
class ANNWrapper:
    """Keras ANN wrapped in a sklearn-style fit/predict interface."""

    def __init__(self, input_dim: int, seed: int = RANDOM_STATE):
        self.input_dim = input_dim
        self.seed      = seed
        self.model_    = None

    def _build(self):
        tf.random.set_seed(self.seed)
        m = keras.Sequential(name="ANN_HAB")
        m.add(keras.layers.Input(shape=(self.input_dim,)))
        # First hidden layer added outside the loop (matches CV analysis script)
        m.add(keras.layers.Dense(ANN_NEURONS, activation=ANN_ACTIVATION))
        m.add(keras.layers.Dropout(ANN_DROPOUT_RATE, seed=self.seed))
        for _ in range(ANN_HIDDEN_LAYERS - 1):
            m.add(keras.layers.Dense(ANN_NEURONS, activation=ANN_ACTIVATION))
            m.add(keras.layers.Dropout(ANN_DROPOUT_RATE, seed=self.seed))
        m.add(keras.layers.Dense(1, activation="sigmoid"))
        m.compile(
            optimizer=keras.optimizers.Adam(learning_rate=ANN_LEARNING_RATE),
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        return m

    def fit(self, X, y):
        self.model_ = self._build()
        cb = [
            keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=ANN_PATIENCE,
                restore_best_weights=True, verbose=0,
            )
        ]
        self.model_.fit(
            X, y,
            epochs=ANN_EPOCHS, batch_size=ANN_BATCH_SIZE,
            validation_split=ANN_VAL_SPLIT,
            callbacks=cb, verbose=0,
        )
        return self

    def predict(self, X):
        proba = self.model_.predict(X, verbose=0).ravel()
        return (proba >= 0.5).astype(int)


# ============================================================================
# CV HELPERS  (all 50 fold scores returned for per-site CI)
# ============================================================================
_SCORING = {
    "accuracy":  make_scorer(accuracy_score),
    "precision": make_scorer(precision_score, zero_division=0),
    "recall":    make_scorer(recall_score,    zero_division=0),
    "f1":        make_scorer(f1_score,        zero_division=0),
}


def cv_all_scores_sklearn(model_builder, X_train_df, y_train,
                          n_splits=CV_N_SPLITS, n_repeats=CV_N_REPEATS):
    """
    Run RepeatedStratifiedKFold CV on the FULL 13-site training set.
    Returns:
        all_mean  : dict[metric -> float]   mean across ALL n_splits*n_repeats folds
        all_scores: dict[metric -> ndarray] all n_splits*n_repeats fold scores
    """
    rskf   = RepeatedStratifiedKFold(
        n_splits=n_splits, n_repeats=n_repeats, random_state=RANDOM_STATE
    )
    model  = model_builder()
    cv_out = cross_validate(model, X_train_df, y_train,
                            cv=rskf, scoring=_SCORING,
                            return_train_score=False, n_jobs=1)

    all_scores = {m: np.array(cv_out[f"test_{m}"]) for m in METRICS}

    # v4: mean across ALL 50 folds (replaces best-repeat mean)
    all_mean = {met: float(all_scores[met].mean()) for met in METRICS}

    return all_mean, all_scores


def cv_all_scores_ann(X_train_sc, y_train,
                      n_splits=CV_N_SPLITS, n_repeats=CV_N_REPEATS):
    """
    Same logic for ANN: manual RepeatedStratifiedKFold on the FULL 13-site set.
    Returns all_mean and all_scores (same structure as cv_all_scores_sklearn).
    """
    rskf        = RepeatedStratifiedKFold(
        n_splits=n_splits, n_repeats=n_repeats, random_state=RANDOM_STATE
    )
    fold_scores = {m: [] for m in METRICS}

    for fold_idx, (tr_idx, val_idx) in enumerate(
            rskf.split(X_train_sc, y_train)):
        Xtr, Xval = X_train_sc[tr_idx], X_train_sc[val_idx]
        ytr, yval = y_train[tr_idx],    y_train[val_idx]
        ann = ANNWrapper(X_train_sc.shape[1], seed=RANDOM_STATE + fold_idx)
        ann.fit(Xtr, ytr)
        yhat = ann.predict(Xval)
        fold_scores["accuracy"].append(accuracy_score(yval, yhat))
        fold_scores["precision"].append(precision_score(yval, yhat, zero_division=0))
        fold_scores["recall"].append(recall_score(yval, yhat, zero_division=0))
        fold_scores["f1"].append(f1_score(yval, yhat, zero_division=0))
        keras.backend.clear_session()
        gc.collect()
        print(f"      ANN CV fold {fold_idx + 1}/{n_splits * n_repeats}")

    all_scores = {m: np.array(fold_scores[m]) for m in METRICS}

    # v4: mean across ALL 50 folds (replaces best-repeat mean)
    all_mean = {met: float(all_scores[met].mean()) for met in METRICS}

    return all_mean, all_scores


# ============================================================================
# MAIN LOSO LOOP
# ============================================================================
def run_loso(X, y, sites):
    """
    For each of the 14 sites (left-out one at a time):
      1.  Train set = all 13 remaining sites (no 70/30 split).
      2.  Run 5-fold × 10-repeat CV on the full 13-site train set.
          Store all 50 per-fold scores → used for per-site training SD.
      3.  Refit final model on the full 13-site train set.
      4.  Predict on left-out site; compute test metrics + bootstrap min–max range.

    Returns
    -------
    results : dict[model_name][site_name] = {
        "train_cv_mean"  : dict[metric -> float],   mean across all 50 CV folds
        "train_cv_lower" : dict[metric -> float],   mean - min across 50 folds
        "train_cv_upper" : dict[metric -> float],   max - mean across 50 folds
        "test_mean"      : dict[metric -> float],   bootstrap mean
        "test_lower"     : dict[metric -> float],   mean - min across bootstrap resamples
        "test_upper"     : dict[metric -> float],   max - mean across bootstrap resamples
    }
    unique_sites : sorted list of site names
    """
    unique_sites = sorted(np.unique(sites))
    assert len(unique_sites) == 14, \
        f"Expected 14 sites, found {len(unique_sites)}: {unique_sites}"

    results = {m: {s: {} for s in unique_sites} for m in ALL_MODELS}

    for loso_idx, left_out in enumerate(unique_sites):
        print(f"\n{'='*72}")
        print(f"LOSO fold {loso_idx + 1}/14  |  Left-out site: {left_out}")
        print(f"{'='*72}")

        mask_train = (sites != left_out)
        mask_test  = (sites == left_out)

        X_13 = X[mask_train]
        y_13 = y[mask_train]
        X_lo = X[mask_test]     # left-out site — ALL rows
        y_lo = y[mask_test]

        # ---- Scale on full 13-site pool ------------------------------------
        scaler   = StandardScaler()
        X_13_sc  = scaler.fit_transform(X_13)
        X_lo_sc  = scaler.transform(X_lo)

        X_13_df  = pd.DataFrame(X_13_sc, columns=FEATURES)
        X_lo_df  = pd.DataFrame(X_lo_sc, columns=FEATURES)

        # ---- sklearn models -----------------------------------------------
        for mname, builder in SKLEARN_BUILDERS.items():
            print(f"  [{mname}] CV on full 13-site training set …")
            all_mean, all_scores = cv_all_scores_sklearn(
                builder, X_13_df, y_13
            )

            # Per-site training min–max error from all 50 CV fold scores
            train_lower = {
                met: mean_minmax_from_scores(all_scores[met])[1]
                for met in METRICS
            }
            train_upper = {
                met: mean_minmax_from_scores(all_scores[met])[2]
                for met in METRICS
            }

            # Final model: fit on full 13-site pool
            final_model = builder()
            final_model.fit(X_13_df, y_13)
            y_pred_lo   = final_model.predict(X_lo_df)

            # Bootstrap mean + min–max range on test set
            boot = bootstrap_minmax(y_lo, y_pred_lo,
                                    seed=RANDOM_STATE + loso_idx)

            results[mname][left_out] = {
                "train_cv_mean":  all_mean,
                "train_cv_lower": train_lower,
                "train_cv_upper": train_upper,
                "test_mean":      {m: boot[m]["mean"]      for m in METRICS},
                "test_lower":     {m: boot[m]["lower_err"] for m in METRICS},
                "test_upper":     {m: boot[m]["upper_err"] for m in METRICS},
            }
            print(f"    CV mean F1={all_mean['f1']:.4f}  "
                  f"Test F1={boot['f1']['mean']:.4f}  "
                  f"Test range [{boot['f1']['mean']-boot['f1']['lower_err']:.4f}"
                  f"–{boot['f1']['mean']+boot['f1']['upper_err']:.4f}]")

        # ---- ANN ----------------------------------------------------------
        print(f"  [ANN] CV on full 13-site training set …")
        all_mean_ann, all_scores_ann = cv_all_scores_ann(X_13_sc, y_13)

        train_lower_ann = {
            met: mean_minmax_from_scores(all_scores_ann[met])[1]
            for met in METRICS
        }
        train_upper_ann = {
            met: mean_minmax_from_scores(all_scores_ann[met])[2]
            for met in METRICS
        }

        # Final ANN: fit on full 13-site pool
        ann_final = ANNWrapper(X_13_sc.shape[1],
                               seed=RANDOM_STATE + loso_idx)
        ann_final.fit(X_13_sc, y_13)
        y_pred_ann   = ann_final.predict(X_lo_sc)
        keras.backend.clear_session()
        gc.collect()

        boot_ann = bootstrap_minmax(y_lo, y_pred_ann,
                                    seed=RANDOM_STATE + loso_idx)

        results["ANN"][left_out] = {
            "train_cv_mean":  all_mean_ann,
            "train_cv_lower": train_lower_ann,
            "train_cv_upper": train_upper_ann,
            "test_mean":      {m: boot_ann[m]["mean"]      for m in METRICS},
            "test_lower":     {m: boot_ann[m]["lower_err"] for m in METRICS},
            "test_upper":     {m: boot_ann[m]["upper_err"] for m in METRICS},
        }
        print(f"    CV mean F1={all_mean_ann['f1']:.4f}  "
              f"Test F1={boot_ann['f1']['mean']:.4f}  "
              f"Test range [{boot_ann['f1']['mean']-boot_ann['f1']['lower_err']:.4f}"
              f"–{boot_ann['f1']['mean']+boot_ann['f1']['upper_err']:.4f}]")

    return results, unique_sites


# ============================================================================
# PLOTTING
# ============================================================================
def plot_model_figure(model_name, results, unique_sites):
    """
    One figure per model.
    4 subplots (Accuracy, Precision, Recall, F1).
    Each subplot: 14 grouped bar pairs.
      - Training bar : mean across all 50 CV folds,
                       asymmetric error bar = [mean-min, max-mean]
      - Test bar     : bootstrap mean,
                       asymmetric error bar = [mean-min, max-mean] across 1000 resamples
    Error bars are naturally bounded within [0, 1].
    """
    color_train = MODEL_COLOR[model_name]
    color_test  = "#AAAAAA"

    n_sites = len(unique_sites)
    x       = np.arange(n_sites)
    bar_w   = 0.35

    fig, axes = plt.subplots(2, 2, figsize=(20, 11))
    fig.suptitle(f"LOSO Performance — {model_name}",
                 fontsize=16, fontweight="bold", y=1.01)
    axes = axes.flatten()

    metric_labels = {
        "accuracy":  "Accuracy",
        "precision": "Precision",
        "recall":    "Recall",
        "f1":        "F1 Score",
    }

    for ax, met in zip(axes, METRICS):
        train_vals  = np.array([
            results[model_name][s]["train_cv_mean"][met]  for s in unique_sites
        ])
        train_lower = np.array([
            results[model_name][s]["train_cv_lower"][met] for s in unique_sites
        ])
        train_upper = np.array([
            results[model_name][s]["train_cv_upper"][met] for s in unique_sites
        ])
        test_vals   = np.array([
            results[model_name][s]["test_mean"][met]      for s in unique_sites
        ])
        test_lower  = np.array([
            results[model_name][s]["test_lower"][met]     for s in unique_sites
        ])
        test_upper  = np.array([
            results[model_name][s]["test_upper"][met]     for s in unique_sites
        ])

        bars_train = ax.bar(
            x - bar_w / 2, train_vals, bar_w,
            color=color_train, alpha=TRAIN_ALPHA,
            label="Training CV (mean of 50 folds, range = min–max)",
            yerr=[train_lower, train_upper],
            error_kw=dict(elinewidth=1.2, capsize=3,
                          ecolor="black", capthick=1.2),
        )
        bars_test = ax.bar(
            x + bar_w / 2, test_vals, bar_w,
            color=color_test, alpha=TEST_ALPHA,
            label="Test — left-out site (bootstrap mean, range = min–max)",
            yerr=[test_lower, test_upper],
            error_kw=dict(elinewidth=1.2, capsize=3,
                          ecolor="dimgray", capthick=1.2),
        )

        # Value labels — position above the upper whisker tip
        for bar, upper in zip(bars_train, train_upper):
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2,
                    h + upper + 0.012,
                    f"{h:.2f}", ha="center", va="bottom",
                    fontsize=6.0, color="black")

        for bar, upper in zip(bars_test, test_upper):
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2,
                    h + upper + 0.012,
                    f"{h:.2f}", ha="center", va="bottom",
                    fontsize=6.0, color="dimgray")

        # Formatting
        ax.set_title(metric_labels[met], fontsize=13, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(unique_sites, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel(metric_labels[met], fontsize=10)
        ax.set_ylim(0, 1.20)
        ax.yaxis.grid(True, linestyle="--", alpha=0.5)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        ax.legend(fontsize=8, loc="upper right")

        ax.annotate(
            "Train: mean of 50 CV folds, whiskers = min–max | Test: bootstrap mean, whiskers = min–max",
            xy=(0.02, 0.02), xycoords="axes fraction",
            fontsize=6.5, color="dimgray", style="italic",
        )

    plt.tight_layout()
    return fig


# ============================================================================
# SAVE SUMMARY TABLE
# ============================================================================
def save_summary_table(results, unique_sites):
    rows = []
    for mname in ALL_MODELS:
        for site in unique_sites:
            r   = results[mname][site]
            row = {"Model": mname, "Left_Out_Site": site}
            for met in METRICS:
                mean = r["train_cv_mean"][met]
                row[f"TrainCV_mean_{met}"]  = mean
                row[f"TrainCV_min_{met}"]   = mean - r["train_cv_lower"][met]
                row[f"TrainCV_max_{met}"]   = mean + r["train_cv_upper"][met]
                t_mean = r["test_mean"][met]
                row[f"Test_mean_{met}"]     = t_mean
                row[f"Test_min_{met}"]      = t_mean - r["test_lower"][met]
                row[f"Test_max_{met}"]      = t_mean + r["test_upper"][met]
            rows.append(row)
    df   = pd.DataFrame(rows)
    path = os.path.join(OUTPUT_DIR, "tables", "loso_performance_summary.csv")
    df.to_csv(path, index=False)
    print(f"\nSummary table saved → {path}")
    return df


# ============================================================================
# ENTRY POINT
# ============================================================================
def main():
    print("TensorFlow:", tf.__version__)
    set_global_seed(RANDOM_STATE)
    setup_dirs()

    print("\nLoading data …")
    X, y, sites, df = load_data()
    print(f"  Rows: {len(y)}  |  Positive class: {y.sum()}  "
          f"({100*y.mean():.1f}%)  |  Sites: {np.unique(sites)}")

    results, unique_sites = run_loso(X, y, sites)

    save_summary_table(results, unique_sites)

    print("\nGenerating figures …")
    for mname in ALL_MODELS:
        print(f"  Plotting: {mname}")
        fig = plot_model_figure(mname, results, unique_sites)
        safe_fname = mname.replace(" ", "_")
        savefig(fig, "loso_performance", f"LOSO_{safe_fname}")
        plt.close(fig)

    print("\nAll done.")


if __name__ == "__main__":
    main()
